In [93]:
%gui qt
import numpy as np
from mayavi import mlab
import matplotlib.pyplot as plt

from time import time

from kifmm_py import (
    KiFmm,
    HelmholtzKernel,
    SingleNodeTree,
    EvalType,
    BlasFieldTranslation,
    FftFieldTranslation,
)

In [94]:
def error(direct, pot):
    return np.abs(direct) - np.abs(pot)

def relative_error(direct, evaluated):
    return np.abs(direct-evaluated)/np.abs(direct)

def ncoeffs(p):
    return 6 * (p-1)**2 + 2

In [95]:
ncoeffs(9) / ncoeffs(6)

2.539473684210526

In [96]:
def rank(k, D):
    return (k*D)**2

In [106]:
rank(wavenumber, 1/2**2)

256.0

In [127]:
np.random.seed(0)

dim = 3
dtype = np.float64
ctype = np.complex128

n_vec = 1
n_sources = 100000
n_targets = 100000
wavenumber = dtype(100)
prune_empty = True

# Setup source/target/charge data in Fortran order
sources = np.random.rand(n_sources * dim).astype(dtype)
targets = np.random.rand(n_targets * dim).astype(dtype)
charges = np.random.rand(n_sources * n_vec).astype(ctype)

eval_type = EvalType.Value

# EvalType computes either potentials (EvalType.Value) or potentials + derivatives (EvalType.ValueDeriv)
kernel = HelmholtzKernel(dtype, wavenumber, eval_type)
# Set FMM Parameters
n_vec = 1
depth=4
tree = SingleNodeTree(sources, targets, charges, depth=depth, prune_empty=prune_empty)

In [128]:
1/wavenumber

0.01

In [129]:
for level in range(2, depth+1):    
    D = 1/(2**level)
    Dk = D * wavenumber
    rank = Dk**2
    print("level", level, "Dk", rank)

level 2 Dk 625.0
level 3 Dk 156.25
level 4 Dk 39.0625


In [130]:
ncoeffs(11)

602

In [137]:
expansion_order = np.array([2, 2, 15, 12, 10], np.uint64)
field_translation = FftFieldTranslation(kernel, block_size=32)
s = time()
fft_fmm = KiFmm(expansion_order, tree, field_translation, timed=True)
print("FFT Setup Time", time()-s)

FFT Setup Time 10.856304168701172


In [138]:
s = time()
fft_fmm.evaluate()
print("FFT Runtime", time()-s)

FFT Runtime 1.6696441173553467


In [139]:
leaf = fft_fmm.leaves_target_tree[0]
evaluated = fft_fmm.leaf_potentials(leaf)
leaf_coords = fft_fmm.coordinates_target_tree(leaf)
nleaf_coords = len(leaf_coords) // dim

direct = np.zeros(nleaf_coords * eval_type.value).astype(ctype)
fft_fmm.evaluate_kernel(EvalType.Value, sources, leaf_coords, charges, direct)
np.mean(relative_error(evaluated, direct))

0.00029195512238265077

In [89]:
expansion_order = np.array([2, 2, 14, 11, 9], np.uint64)
svd_threshold = 1e-8
field_translation = BlasFieldTranslation(kernel, svd_threshold, surface_diff=1, random=False)

In [90]:
s = time()
blas_fmm = KiFmm(expansion_order, tree, field_translation, timed=True)
print("BLAS Setup Time", time()-s)

BLAS Setup Time 291.32842111587524


In [91]:
s = time()
blas_fmm.evaluate()
print("BLAS Runtime", time()-s)

BLAS Runtime 2.4422810077667236


In [92]:
leaf = blas_fmm.leaves_target_tree[0]
evaluated = blas_fmm.leaf_potentials(leaf)
leaf_coords = blas_fmm.coordinates_target_tree(leaf)
nleaf_coords = len(leaf_coords) // 3

direct = np.zeros(nleaf_coords * eval_type.value).astype(ctype)
blas_fmm.evaluate_kernel(EvalType.Value, sources, leaf_coords, charges, direct)
np.mean(relative_error(evaluated, direct))

6.50582606092319e-09